# Dubai Building Code - FAISS Index Builder
Embeds `dubai_chunks.json` using `nomic-embed-text-v1` and produces:
- `index.faiss` - FAISS IndexFlatIP (cosine via inner product on L2-normalised vectors)
- `index_meta.pkl` - chunk metadata list, including v2 fields such as `chunk_type`, `table`, `subject`, and `table_type`

**Runtime:** GPU (A100 recommended). Upload the v2 `dubai_chunks.json` from `indexing_v2/` when prompted, then download the two output files into `test/` or `deployment/`.


In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install -q sentence-transformers faiss-cpu einops

In [ ]:
# ── 2. Upload dubai_chunks.json ───────────────────────────────────────────────
from google.colab import files
print('Upload dubai_chunks.json ...')
uploaded = files.upload()
assert 'dubai_chunks.json' in uploaded, 'Please upload dubai_chunks.json'
print('Uploaded OK')

In [ ]:
# ── 3. Load chunks ────────────────────────────────────────────────────────────
import json
with open('dubai_chunks.json', encoding='utf-8') as f:
    chunks = json.load(f)

print(f'Chunks loaded : {len(chunks)}')
print(f'Sample text   : {chunks[10]["text"][:120]}...')

In [ ]:
# ── 4. Load nomic-embed-text-v1 ───────────────────────────────────────────────
# Same model that Ollama serves as 'nomic-embed-text' — vectors are compatible.
import torch
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

model = SentenceTransformer(
    'nomic-ai/nomic-embed-text-v1',
    trust_remote_code=True,
    device=device,
)
print('Model loaded')

In [ ]:
# ── 5. Embed all chunks ───────────────────────────────────────────────────────
# Prefix must match build_index.py: 'search_document: {text}'
import numpy as np
from tqdm.notebook import tqdm

BATCH_SIZE = 64

texts = [f"search_document: {c['text']}" for c in chunks]

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,   # L2-normalise for cosine via inner product
    device=device,
)

embeddings = np.array(embeddings, dtype='float32')
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# ── 6. Build FAISS index ──────────────────────────────────────────────────────
import faiss

dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)   # inner product on normalised vecs = cosine
index.add(embeddings)

faiss.write_index(index, 'index.faiss')
print(f'FAISS index: {index.ntotal} vectors, dim={dim}')

In [ ]:
# -- 7. Save metadata ----------------------------------------------------------
import pickle

# Preserve all chunk metadata. The typed retriever needs fields such as
# chunk_type='text'/'fact', table, subject, and table_type.
meta = [dict(c) for c in chunks]

required = {'id', 'text', 'section', 'page_start', 'page_end', 'chunk_type'}
missing = sorted(required - set(meta[0]))
if missing:
    raise ValueError(
        f'Metadata is missing required v2 fields: {missing}. '
        'Upload indexing_v2/dubai_chunks.json, not the old indexing/dubai_chunks.json.'
    )

with open('index_meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

type_counts = {t: sum(1 for c in meta if c.get('chunk_type') == t) for t in ('text', 'fact')}
print(f'Metadata saved: {len(meta)} entries')
print(f'Chunk types   : {type_counts}')


In [ ]:
# -- 8. Quick sanity check -----------------------------------------------------
query = 'search_query: what are the requirements for bedrooms'
q_vec = model.encode([query], normalize_embeddings=True)
scores, indices = index.search(np.array(q_vec, dtype='float32'), k=5)

print('Top 5 vector results for:', query)
for score, idx in zip(scores[0], indices[0]):
    c = meta[idx]
    print(f'  score={score:.3f} | {c.get("chunk_type")} | p{c["page_start"]}-{c["page_end"]} | {c["section"]}')
    print(f'  {c["text"][:180]}...')
    print()

bedroom_facts = [
    c for c in meta
    if c.get('chunk_type') == 'fact'
    and 'Residential living space (bedroom, living room)' in c.get('text', '')
    and '10.5' in c.get('text', '')
]
assert bedroom_facts, 'Bedroom Table B.3 fact chunk not found in metadata.'
print('Bedroom fact sanity check OK:')
print(bedroom_facts[0]['text'])


In [ ]:
# -- 9. Download outputs -------------------------------------------------------
# Copy both files into test/ for local testing, or deployment/ for the cloud app.
files.download('index.faiss')
files.download('index_meta.pkl')
